# BM25 Lexical Search (Lesson 37)

Follow-along notebook for *BM25 lexical search* in Building with the Claude API.

The lesson uses a pre-built `BM25Index` class. It's defined here, written to be readable so you can see how BM25 actually scores documents.

**Sections 1-5 need no API keys.** Section 6 compares against semantic search and needs `VOYAGE_API_KEY`.

**Run order:** Run All after a kernel restart.

## 1. Setup and the sample report
Uses `report.md` from the RAG flow notebook, or recreates it if it's missing.

In [1]:
import math
import re
from collections import Counter
from pathlib import Path

REPORT = r'''# Annual Interdisciplinary Research Review 2023

## Executive Summary
This review summarizes the year's work across five research and operations divisions. Highlights include a successful Phase II clinical trial, a major platform release from software engineering, improved financial performance, and the resolution of a significant security incident. Each section below covers goals, results, and next steps.

## Table of Contents
1. Methodology
2. Section 1: Medical Research
3. Section 2: Software Engineering
4. Section 3: Financial Analysis
5. Section 4: Cybersecurity
6. Section 5: Legal and Compliance
7. Conclusion

## Methodology
Each division reported quarterly metrics using a shared template. Teams documented their processes, tools, and outcomes, and an independent review panel validated the figures. Engineering-heavy divisions also submitted system logs and release notes, which were cross-checked against project tracking data.

## Section 1: Medical Research
The medical research team completed Phase II trials for compound NR-204, a treatment for chronic inflammation. Enrollment paused briefly after a stomach bug spread through one trial site; the bug was traced to a catering supplier and resolved within two weeks. Final results showed a 34 percent reduction in inflammation markers, and Phase III planning has begun.

## Section 2: Software Engineering
The software engineering department shipped four major releases of the analytics platform. Engineers fixed 1,247 bugs, a 22 percent increase over the prior year, and reduced average resolution time from nine days to four. The team migrated core services to a new container platform and raised automated test coverage to 81 percent.

## Section 3: Financial Analysis
Total revenue reached 96 million dollars, up 18 percent. Operating margin improved to 14 percent on lower infrastructure costs. Research spending totaled 21 million dollars, most of it in medical research. Cash reserves at year end were 38 million dollars.

## Section 4: Cybersecurity
The security team handled 37 incidents during the year, most of them low severity. The most significant, incident INC-2023-Q4-011, involved unauthorized access to a staging database through a misconfigured API gateway. The team contained it within six hours, rotated all affected credentials, and found no evidence that customer data left the environment. A post-incident review led to mandatory configuration audits for all gateways.

## Section 5: Legal and Compliance
The legal team renewed data processing agreements with all major vendors and completed an external privacy audit with no material findings. Following incident INC-2023-Q4-011, counsel confirmed that no regulatory notification was required.

## Conclusion
The organization ends the year in a strong position. Priorities for next year include Phase III trials, expanding the analytics platform into healthcare, and continued investment in security controls.
'''

if not Path("report.md").exists() or "INC-2023-Q4-011" not in Path("report.md").read_text(encoding="utf-8"):
    Path("report.md").write_text(REPORT, encoding="utf-8")

text = Path("report.md").read_text(encoding="utf-8")


def chunk_by_section(document_text):
    return [c.strip() for c in re.split(r"\n(?=## )", document_text) if c.strip()]


chunks = chunk_by_section(text)
for i, c in enumerate(chunks):
    print(i, c.splitlines()[0])

0 # Annual Interdisciplinary Research Review 2023
1 ## Executive Summary
2 ## Table of Contents
3 ## Methodology
4 ## Section 1: Medical Research
5 ## Section 2: Software Engineering
6 ## Section 3: Financial Analysis
7 ## Section 4: Cybersecurity
8 ## Section 5: Legal and Compliance
9 ## Conclusion


## 2. Step 1: Tokenizing

BM25 works on terms, so the first job is splitting text into them. The tokenizer matters a lot: a naive split on non-letters would break `INC-2023-Q4-011` into `inc`, `2023`, `q4`, `011`, and the ID would stop being a single rare term.

This one lowercases and keeps hyphenated alphanumeric runs together.

In [2]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*", text.lower())


print(tokenize("a INC-2023-Q4-011"))
print(tokenize("What happened with INC-2023-Q4-011?"))
print(re.findall(r"[a-z0-9]+", "INC-2023-Q4-011".lower()), "<- what a naive tokenizer would do")

['a', 'inc-2023-q4-011']
['what', 'happened', 'with', 'inc-2023-q4-011']
['inc', '2023', 'q4', '011'] <- what a naive tokenizer would do


## 3. The `BM25Index` class

For each query term, BM25 combines:
- **IDF (inverse document frequency):** rare terms are worth more. A term in every chunk scores near zero; a term in one chunk scores high. That's the lesson's steps 2 and 3.
- **Term frequency, with saturation (`k1`):** more occurrences help, but with diminishing returns, so repeating a word 50 times doesn't win.
- **Length normalization (`b`):** a match in a short chunk counts more than the same match in a very long one.

A document's score is the sum over query terms. Higher is better.

> The lesson's loop names the second value `distance`, but BM25 returns a **score** where higher means more relevant, the opposite of the cosine distance from the vector store. The next lesson merges the two by rank, which sidesteps the mismatch.

In [3]:
class BM25Index:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.documents = []
        self.doc_tokens = []      # tokenized content per document
        self.doc_freq = Counter() # term -> number of documents containing it

    def add_document(self, document):
        tokens = tokenize(document["content"])
        self.documents.append(document)
        self.doc_tokens.append(tokens)
        self.doc_freq.update(set(tokens))

    def __len__(self):
        return len(self.documents)

    @property
    def avg_doc_len(self):
        return sum(len(t) for t in self.doc_tokens) / len(self.doc_tokens)

    def idf(self, term):
        n = len(self.documents)
        df = self.doc_freq.get(term, 0)
        return math.log((n - df + 0.5) / (df + 0.5) + 1)

    def score(self, query_tokens, doc_index):
        tokens = self.doc_tokens[doc_index]
        counts = Counter(tokens)
        length_norm = 1 - self.b + self.b * len(tokens) / self.avg_doc_len
        total = 0.0
        for term in query_tokens:
            tf = counts.get(term, 0)
            if tf:
                total += self.idf(term) * tf * (self.k1 + 1) / (tf + self.k1 * length_norm)
        return total

    def search(self, query, k=3):
        query_tokens = tokenize(query)
        scored = [(doc, self.score(query_tokens, i)) for i, doc in enumerate(self.documents)]
        scored.sort(key=lambda pair: pair[1], reverse=True)
        return scored[:k]

    def explain(self, query):
        """Show each query term's document frequency and IDF weight."""
        print(f"{'term':<18} {'in docs':>7} {'idf':>6}")
        for term in dict.fromkeys(tokenize(query)):
            print(f"{term:<18} {self.doc_freq.get(term, 0):>7} {self.idf(term):>6.2f}")

## 4. The lesson's code

In [4]:
# 1. Chunk the text by sections (done above)

# 2. Create a BM25 store and add documents
store = BM25Index()
for chunk in chunks:
    store.add_document({"content": chunk})

# 3. Search the store
results = store.search("What happened with INC-2023-Q4-011?", 3)

for doc, score in results:
    print(f"{score:.3f}\n{doc['content'][:200]}\n----\n")

4.536
## Section 5: Legal and Compliance
The legal team renewed data processing agreements with all major vendors and completed an external privacy audit with no material findings. Following incident INC-20
----

1.183
## Section 4: Cybersecurity
The security team handled 37 incidents during the year, most of them low severity. The most significant, incident INC-2023-Q4-011, involved unauthorized access to a staging
----

0.000
# Annual Interdisciplinary Research Review 2023
----



Both sections that mention the incident should rank on top (in this sample report, Cybersecurity and Legal and Compliance; in the lesson's report it's Software Engineering and Cybersecurity). Every other section scores **0**, because none contains the ID and the remaining words are too common to matter.

## 5. Why: term weights

This is the lesson's step 3 made visible. Read the table carefully, because it shows two things the lesson glosses over:

- **Terms in no chunk** (`what`, `happened`) have a high IDF but contribute **nothing**, since there's nothing to match.
- **Rarity is relative to *your* corpus.** In this small report, `with` happens to appear in only one chunk (Legal), so it gets a *higher* weight than the incident ID, which appears in two. That's why Legal outscores Cybersecurity here: it matches both terms, and it's also shorter, so length normalization boosts it. In a large corpus, `with` would be everywhere and its weight would fall toward zero.

That's also why many BM25 setups strip stopwords before indexing.

In [5]:
store.explain("What happened with INC-2023-Q4-011?")

term               in docs    idf
what                     0   3.09
happened                 0   3.09
with                     1   1.99
inc-2023-q4-011          2   1.48


Try a few of your own. Words that appear in many sections barely move the score; distinctive words dominate:

In [6]:
store.explain("software engineering bugs fixed")
print()
for doc, score in store.search("software engineering bugs fixed", 3):
    print(f"{score:.3f}  {doc['content'].splitlines()[0]}")

term               in docs    idf
software                 3   1.15
engineering              3   1.15
bugs                     1   1.99
fixed                    1   1.99

6.389  ## Section 2: Software Engineering
2.572  ## Table of Contents
2.117  ## Executive Summary


## 6. Compare with semantic search (needs `VOYAGE_API_KEY`)

Same chunks, same questions, both methods side by side. The first two questions are exact lookups; the last two are paraphrases with little word overlap. Expect each method to win where the other is weak, which is the case for hybrid search in the next lesson.

In [7]:
import voyageai
from dotenv import load_dotenv

load_dotenv()
voyage_client = voyageai.Client(max_retries=3)
EMBED_MODEL = "voyage-3-large"


def cosine_distance(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    return 1 - dot / (math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b)))


chunk_embeddings = voyage_client.embed(chunks, model=EMBED_MODEL, input_type="document").embeddings


def semantic_search(query, k=3):
    q = voyage_client.embed([query], model=EMBED_MODEL, input_type="query").embeddings[0]
    scored = sorted(((cosine_distance(q, e), c) for e, c in zip(chunk_embeddings, chunks)))
    return scored[:k]


questions = [
    "What happened with INC-2023-Q4-011?",
    "NR-204",
    "How profitable was the company?",
    "Did anything go wrong in the clinical studies?",
]

for q in questions:
    print(f"Q: {q}")
    sem = [c.splitlines()[0] for _, c in semantic_search(q)]
    lex = [(d["content"].splitlines()[0], s) for d, s in store.search(q)]
    for i in range(3):
        lex_label = f"{lex[i][0]} ({lex[i][1]:.2f})" if lex[i][1] > 0 else "(no match)"
        print(f"   {i+1}. semantic: {sem[i]:<38} BM25: {lex_label}")
    print()

c:\Users\bspedden\OneDrive - The Select Group LLC\Documents\work\Building-With-The-Claude-API\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Q: What happened with INC-2023-Q4-011?
   1. semantic: ## Section 5: Legal and Compliance     BM25: ## Section 5: Legal and Compliance (4.54)
   2. semantic: ## Section 4: Cybersecurity            BM25: ## Section 4: Cybersecurity (1.18)
   3. semantic: ## Section 1: Medical Research         BM25: (no match)

Q: NR-204
   1. semantic: ## Section 1: Medical Research         BM25: ## Section 1: Medical Research (1.66)
   2. semantic: ## Section 2: Software Engineering     BM25: (no match)
   3. semantic: ## Section 4: Cybersecurity            BM25: (no match)

Q: How profitable was the company?


RateLimitError: You have not yet added your payment method in the billing page and will have reduced rate limits of 3 RPM and 10K TPM. To unlock our standard rate limits, please add a payment method in the billing page for the appropriate organization in the user dashboard (https://dashboard.voyageai.com/). Even with payment methods entered, the free tokens (200M tokens for Voyage series 3) will still apply. After adding a payment method, you should see your rate limits increase after several minutes. See our pricing docs (https://docs.voyageai.com/docs/pricing) for the free tokens for your model.

What to look for:
- **Exact IDs** (`INC-2023-Q4-011`, `NR-204`): BM25 ranks precisely the sections containing them; semantic search may mix in sections that are merely on a related topic.
- **Paraphrases** ("profitable", "clinical studies"): the report says "operating margin" and "trials", so BM25 finds little or nothing, while semantic search should still land on Financial Analysis and Medical Research.

## Takeaways
- BM25 scores exact term matches, weighting rare terms heavily and common terms barely at all.
- Your tokenizer decides what a "term" is; get it wrong and IDs or codes stop matching.
- Lexical and semantic search fail in opposite situations, so production RAG usually runs both and merges the results.
- In production you'd typically use a library (`rank_bm25`) or a search engine (Elasticsearch, OpenSearch) rather than this class, but the scoring is the same idea.